# BiCGSTAB — Interactive Experiments

Compare the project's right-preconditioned BiCGSTAB with SciPy on nonsymmetric systems.

Select settings and click **Run experiment**. All matrices use a fixed random seed for reproducibility.


In [2]:
from pathlib import Path
import sys
import time

# Discover the checkout root in local Jupyter and Binder sessions.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Launch Jupyter from inside the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from krylov_methods.notebooks.notebook_utils import (
    spd_problem, nonsymmetric_problem, residual_plot, error_plot, report,
)

from scipy.sparse.linalg import bicgstab as scipy_bicgstab
from krylov_methods.bicgstab_solver import solve_bicgstab

from scipy.sparse.linalg import LinearOperator
from krylov_methods.preconditioners import make_preconditioner


## Interactive experiment

The curves show the *true* residual $\|b-Ax_k\|_2$. The dashed line is the absolute/relative stopping threshold.

In [4]:
system = widgets.Dropdown(
    options=[("Nonsymmetric", "nonsymmetric"), ("Real SPD", "spd")],
    description="System",
)
size = widgets.IntSlider(value=30, min=5, max=100, step=5, description="Size n")
condition = widgets.FloatLogSlider(value=1e4, base=10, min=1, max=6, step=1, description="Cond(A), SPD")
tol = widgets.FloatLogSlider(value=1e-9, base=10, min=-12, max=-5, step=1, description="rtol")
budget = widgets.IntSlider(value=200, min=20, max=600, step=20, description="Max iterations")
preconds = widgets.Dropdown(
    options=["none", "jacobi", "ilu", "lu", "ssor", "cholesky", "ic0", "custom (diagonal)"],
    value="none",
    description="Precond",
    layout=widgets.Layout(width="350px"),
)
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        A, b, x_true = (
            nonsymmetric_problem(size.value)
            if system.value == "nonsymmetric"
            else spd_problem(size.value, condition.value)
        )
        threshold = max(1e-12, tol.value * np.linalg.norm(b))
        eigs = np.linalg.eigvals(A)
        print("SYSTEM INFORMATION")
        print(f"Matrix: {system.value}; dimension={size.value}; dense; dtype={A.dtype}")
        print(f"Symmetry/Hermitian error ||A-Aᴴ||₂={np.linalg.norm(A-A.conj().T, ord=2):.3e}")
        print(f"Condition number κ₂(A)={np.linalg.cond(A):.3e}")
        print(f"Eigenvalue real parts: [{eigs.real.min():.3e}, {eigs.real.max():.3e}]")
        print(f"Eigenvalue imaginary parts: [{eigs.imag.min():.3e}, {eigs.imag.max():.3e}]")
        print(f"||b||₂={np.linalg.norm(b):.3e}; ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"Initial guess x₀=0; initial residual={np.linalg.norm(b):.3e}")
        print(f"True-residual stopping threshold={threshold:.3e}")
        print(f"Iteration budget={budget.value}; BiCGSTAB uses RIGHT preconditioning.\n")

        histories = {}
        error_histories = {}
        for name in (preconds.value,):
            if name in ("cholesky", "ic0") and system.value != "spd":
                print(f"Skipping {name}: requires a real SPD system.")
                continue
            custom = name == "custom (diagonal)"
            p = "custom" if custom else None if name == "none" else name
            M = (lambda v: v / np.diag(A)) if custom else None

            start = time.perf_counter()
            result = solve_bicgstab(
                A, b, rtol=tol.value, atol=1e-12, max_iter=budget.value,
                preconditioner=p, M=M, store_history=True,
            )
            elapsed = time.perf_counter() - start
            report("Our BiCG: " + name, result.solution, x_true, b, A,
                   result.iterations, result.converged)
            print(f"  Solver-call time={elapsed:.6f} s; status={result.message}")
            if result.breakdown:
                print("  Breakdown:", result.message)
            histories["BiCGSTAB / " + name] = result.residual_norms
            error_histories["BiCGSTAB / " + name] = [
                np.linalg.norm(xk - x_true) / np.linalg.norm(x_true)
                for xk in result.history
            ]

        # Compare SciPy with each selected preconditioner, not only None.
        for name in (preconds.value,):
            if name in ("cholesky", "ic0") and system.value != "spd":
                continue
            custom = name == "custom (diagonal)"
            kind = "custom" if custom else None if name == "none" else name
            custom_M = (lambda v: v / np.diag(A)) if custom else None
            inverse_action = make_preconditioner(A, kind, M=custom_M)
            scipy_M = LinearOperator(A.shape, matvec=inverse_action, dtype=A.dtype)
            scipy_history = [np.linalg.norm(b)]
            scipy_errors = [1.0]

            def scipy_callback(xk):
                scipy_history.append(np.linalg.norm(b - A @ xk))
                scipy_errors.append(np.linalg.norm(xk - x_true) / np.linalg.norm(x_true))

            start = time.perf_counter()
            x, info = scipy_bicgstab(
                A, b, M=scipy_M, rtol=tol.value, atol=1e-12,
                maxiter=budget.value, callback=scipy_callback,
            )
            elapsed = time.perf_counter() - start
            final_norm = np.linalg.norm(b - A @ x)
            # SciPy may converge with an alpha-only final step; its callback
            # is not guaranteed to run for that last update.
            if not np.isclose(scipy_history[-1], final_norm,
                              rtol=1e-12, atol=1e-14):
                scipy_history.append(final_norm)
                scipy_errors.append(np.linalg.norm(x-x_true) / np.linalg.norm(x_true))
            report("SciPy: " + name, x, x_true, b, A,
                   len(scipy_history)-1, info == 0)
            print(f"  SciPy time={elapsed:.6f} s; info={info}")
            histories["SciPy / " + name] = scipy_history
            error_histories["SciPy / " + name] = scipy_errors

        residual_plot(histories, threshold, "BiCGSTAB — True residual convergence")
        error_plot(error_histories, "BiCGSTAB — Relative solution-error convergence")
        print("Interpretation: residual curves may oscillate. A small residual "
              "does not always imply small forward error when A is ill-conditioned.")
        print("Timing includes built-in preconditioner setup and iteration costs.")


button.on_click(run)
display(widgets.VBox([
    system, size, condition, tol, budget, preconds, button, output,
]))


### Notes

Small residual does not always imply small solution error for ill-conditioned systems. Compare both metrics; a preconditioner changes convergence but should not change the exact solution.


## Numerical exploration — Nonmonotonic residuals

Unlike GMRES, BiCGSTAB does **not** globally minimize the residual norm at
each iteration. The stabilization coefficient $\omega_k$ minimizes the
norm of an **intermediate** residual along one direction, but full residual
histories may increase, oscillate, or stagnate.

Use the following deliberately nonsymmetric, nonnormal upper-bidiagonal
system to investigate this behavior. A smaller residual does not by itself
guarantee smaller solution error, particularly when the matrix is poorly
conditioned. A breakdown or failure to converge is a numerical outcome,
not proof that the linear system has no solution.


In [7]:
nonnormality = widgets.FloatSlider(
    value=3.0, min=0.5, max=8.0, step=0.5, description="Upper coupling"
)
nonmonotonic_button = widgets.Button(description="Explore nonmonotonic convergence", button_style="info")
nonmonotonic_output = widgets.Output()

def run_nonmonotonic(_):
    with nonmonotonic_output:
        clear_output(wait=True)
        n = 25
        A = 2 * np.eye(n) + nonnormality.value * np.eye(n, k=1)
        A += 0.3 * np.eye(n, k=-1)
        x_true = np.ones(n)
        b = A @ x_true
        result = solve_bicgstab(A, b, max_iter=150, rtol=1e-10,
                                atol=1e-12, store_history=True)
        residuals = np.asarray(result.residual_norms)
        rises = int(np.count_nonzero(np.diff(residuals) > 0))
        print(f"Upper coupling={nonnormality.value:g}; cond₂(A)={np.linalg.cond(A):.3e}")
        report("BiCGSTAB", result.solution, x_true, b, A,
               result.iterations, result.converged)
        print(f"Residual increases observed: {rises}; status: {result.message}")
        if result.breakdown:
            print("Numerical breakdown detected.")
        residual_plot({"BiCGSTAB": residuals},
                      max(1e-12, 1e-10*np.linalg.norm(b)),
                      "BiCGSTAB residual fluctuations on a nonnormal system")
        errors = [np.linalg.norm(xk-x_true)/np.linalg.norm(x_true)
                  for xk in result.history]
        error_plot({"BiCGSTAB": errors}, "BiCGSTAB forward error on a nonnormal system")

nonmonotonic_button.on_click(run_nonmonotonic)
display(widgets.VBox([nonnormality, nonmonotonic_button, nonmonotonic_output]))
